In [1]:
import json
import re

In [2]:
with open(r'./data\protokolle\16_071_2006-12-01.json') as f:
    d = json.load(f)
    text = d['text']
    text = re.sub(r'BÜNDNIS(?:SES)?\s*90\/DIE\s*GRÜNEN', 'GRÜNE', text) # That's a long short form for a party...

In [3]:
name_match = r'((?:Dr\.\s)?(?:\w+(?:-\w+)?\s)+\w+(?:-\w+)?|\w+(?:\s\w+)*)'
parties = ['CDU/CSU', 'GRÜNE','SPD', 'FDP', 'AFD', 'DIE LINKE', 'KPD', 'BP', 'DP', 'WAV', 'Z' ] # Die Abkürzungen der wichtigsten - auch historischen - Parteien
partei_match = r'(CDU\/CSU|CSU|CDU|GRÜNE|FDP|AfD|SPD|KPD|BP|DP|WAV|Z|DIE LINKE)' # Parteilos + alte Parteien fehlen!
kommentar_match = rf'{name_match}\s+\[{partei_match}\]: (.*?)[–\)]'
beifall_match = r'[(?:–\s)\(]Beifall .*?[–\)]'
#beifall_match1 = r'\(Beifall .*?[–\)]'
zuruf_match = rf'[(?:–\s)\(]Zuruf .*?{partei_match}: (.*?)[–\)]'
speaker_match = rf'{name_match}\s\({partei_match}\):|\n\n\n{name_match}, (.*?):'


### Get List of all Representatives and their party affiliation since 1949

In [4]:
import xml.etree.ElementTree as ET

In [48]:
tree = ET.parse('./data/MDB_STAMMDATEN.XML')
root = tree.getroot()
bt_list = []
# dict for saving in json
for mdb in root.findall('MDB'):
    dict = {
        'last_name':mdb.find('.//NACHNAME').text,
        'first_name':mdb.find('.//VORNAME').text,
        'anrede':mdb.find('.//ANREDE_TITEL').text,
        'party':mdb.find('.//PARTEI_KURZ').text,
        'election_period':mdb.find('.//WP').text
    }
    if dict['party'] == 'BÜNDNIS 90/DIE GRÜNEN':
        dict['party'] = "GRÜNE"
    bt_list.append(dict)

bt_tuples = [(f"{adbt['first_name']} {adbt['last_name']}", adbt['party'], adbt['election_period']) for adbt in bt_list]
bt_tuples[:5]

[('Manfred Abelein', 'CDU', '5'),
 ('Ernst Achenbach', 'FDP', '3'),
 ('Annemarie Ackermann', 'CDU', '2'),
 ('Else Ackermann', 'CDU', '11'),
 ('Ulrich Adam', 'CDU', '12')]

In [49]:
def find_party_by_name(name, tuple_list):
    if name.startswith('Dr.'):
        name = name[4:]
    for item in tuple_list:
        if name in item[0]:
            return item[1]
    return "Party not found"

In [50]:
print(find_party_by_name('Dr. Robert Habeck', bt_tuples))

GRÜNE


In [45]:
speeches_raw = re.split(speaker_match, text)[1:]
speeches = []
for i in range(0, len(speeches_raw), 5):
    if speeches_raw[i] != None and speeches_raw[i+1] != None:
        name = speeches_raw[i].strip()    
        party = speeches_raw[i+1].strip()
    else:
        name = speeches_raw[i+2].strip()
        party = find_party_by_name(name, bt_tuples)
        if party == "Party not found":
            print(f'Party not found for {name}')
    
    speech = {
        'speaker':{
            'name':name,
            'party':party
        },
        'text': re.split(r'(:\n)|(\nAnlage)',speeches_raw[i+4])[0].strip() # handle end of File and sometimes the speakers get interrupted by the Bundestagspräsident or Vice Bundestagspräsident
        #'text': speeches_raw[i+4].strip()
    }
    speeches.append(speech)


In [10]:
for speech in speeches:
    comments = []
    # comments with known speaker
    for match in re.finditer(kommentar_match,speech['text']):
        comment = {
            'commentator': {
                'name': match.group(1),
                'party': match.group(2)
            },
            'text': match.group(3),
            'preceding_context': speech['text'][:match.start()] 
        }
        comments.append(comment)
    
    # comments with unknown speaker
    for match in re.finditer(zuruf_match, re.sub('der LINKEN', 'DIE LINKE', speech['text'])):
        comment = {'commentator':{
                'name': '<unknown>',
                'party': match.group(1)
            },
            'text': match.group(2),
            'preceding_context': speech['text'][:match.start()]
        }
        comments.append(comment)

    speech['comments'] = comments

    # applause
    beifall = re.findall(beifall_match, speech['text'])
    beifall = ''.join(beifall)
    beifall = re.sub('der LINKEN', 'DIE LINKE', beifall)
    beifall = re.sub('CSU|CDU', r'CDU\/CSU', beifall) # not counting twice...
    beifall_counts = {party: beifall.count(f' {party}') for party in parties}
    speech['applause'] = beifall_counts

In [11]:
json_string = json.dumps(speeches, indent=4) 

# Write JSON string to a file
with open("parsed_example.json", "w") as json_file:
    json_file.write(json_string)

# For all Plenarprotokolle in a given folder

In [12]:
import os

In [13]:
data_path = './data/protokolle/'

In [55]:
for protokoll in os.listdir(data_path):
    print(f"filepath: {data_path+protokoll}")
    with open(data_path+protokoll) as f:
        doc = json.load(f)
        text = doc['text']

    speeches_raw = re.split(speaker_match, text)[1:]
    speeches = []
    for i in range(0, len(speeches_raw), 5):
        if speeches_raw[i] != None and speeches_raw[i+1] != None:
            name = speeches_raw[i].strip()    
            party = speeches_raw[i+1].strip()
        else:
            name = speeches_raw[i+2].strip()
            party = find_party_by_name(name, bt_tuples)
            if party == "Party not found":
                print(f'Party not found for {name}')

        speech = {
            'speaker':{
                'name':name,
                'party':party
            },
            'text': re.split(r'(\nVizepräs.{0,99}?:)|(\nPräsid.{0,99}?:)|(:\n)',speeches_raw[i+4])[0].strip() # sometimes the speakers get interrupted by the Bundestagspräsident or Vice Bundestagspräsident
            #'text': re.split(r'(:\n)|(\nAnlage)',speeches_raw[i+4])[0].strip()
        }

        ################
        # comments
        ################

        comments = []
        # comments with known speaker
        for match in re.finditer(kommentar_match,speech['text']):
            comment = {
                'commentator': {
                    'name': match.group(1),
                    'party': match.group(2)
                },
                'text': match.group(3),
                'preceding_context': speech['text'][:match.start()] 
            }
            comments.append(comment)
        
        # comments with unknown speaker
        for match in re.finditer(zuruf_match, re.sub('der LINKEN', 'DIE LINKE', speech['text'])):
            comment = {'commentator':{
                    'name': '<unknown>',
                    'party': match.group(1)
                },
                'text': match.group(2),
                'preceding_context': speech['text'][:match.start()]
            }
            comments.append(comment)

        speech['comments'] = comments

        ################
        # applause
        ################

        beifall = re.findall(beifall_match, speech['text'])
        beifall = ''.join(beifall)
        beifall = re.sub('der LINKEN', 'DIE LINKE', beifall)
        beifall_counts = {party: beifall.count(f' {party}') for party in parties}
        speech['applause'] = beifall_counts

        speeches.append(speech)

    json_string = json.dumps(speeches, indent=4) 
    path = f'./data/parsed_comments/{doc['wahlperiode']}_{doc['dokumentnummer'].split(r'/')[1].zfill(3)}_{doc['datum']}_parsed.json'
    # Write JSON string to a file
    with open(path, "w") as json_file:
        json_file.write(json_string)   


filepath: ./data/protokolle/16_071_2006-12-01.json
filepath: ./data/protokolle/20_010_2022-01-12.json
Party not found for Nancy Faeser
Party not found for Nancy Faeser
filepath: ./data/protokolle/20_011_2022-01-13.json
Party not found for Anne Spiegel
Party not found for Klara Geywitz
filepath: ./data/protokolle/20_012_2022-01-14.json
filepath: ./data/protokolle/20_148_2024-01-19.json


In [15]:
kommentare = re.findall(kommentar_match, text)
kommentare

[('Andreas Bleck',
  'AfD',
  'Ihr habt Deutschland abgewirtschaftet! Ihr werdet bei der nächsten Bundestagswahl abgestraft!\xa0'),
 ('Albrecht Glaser', 'AfD', 'Dreckiger Schmutz!'),
 ('Andreas Bleck', 'AfD', 'Das macht ihr! Noch nie ging es uns so schlecht!'),
 ('Andreas Bleck', 'AfD', 'Das entscheiden immer noch die Wähler!'),
 ('Andreas Bleck', 'AfD', 'Vor allem!'),
 ('Alexander Dobrindt', 'CDU/CSU', 'So ist es!'),
 ('Jakob Blankenburg',
  'SPD',
  'Wer war denn die letzten Jahre Landwirtschaftsminister?'),
 ('Bettina Hagedorn', 'SPD', 'Bei Ihnen passt gar nichts zusammen!'),
 ('Dr.\xa0Rainer Kraft', 'AfD', 'Von „Correctiv“!'),
 ('Dr.\xa0Rainer Kraft', 'AfD', 'Unbewiesene Behauptung!'),
 ('Dr.\xa0Rainer Kraft',
  'AfD',
  'Das haben sie vor 20\xa0Jahren auch schon gesagt!'),
 ('Dr.\xa0Rainer Kraft',
  'AfD',
  'Wozu brauchen die Eisbrecher, wenn das Eis weg ist?'),
 ('Johannes Schraps', 'SPD', 'Sehr richtig!'),
 ('Dr.\xa0Karamba Diaby', 'SPD', 'Halb voll!'),
 ('Dr.\xa0Karamba Diaby'

In [16]:
import json

In [17]:
with open(r'./data\protokolle\20_012_2022-01-14.json') as f:
    d = json.load(f)
    text = d['text']

In [18]:
with open("./testtext2.txt" , 'w', encoding='utf8') as f:
    f.write(text)

In [19]:
satzende_match = '(?<!\b(?:Dr|med|z\.B|etc)).\s+(?=[A-Z])'

<>:1: SyntaxWarning: invalid escape sequence '\.'
<>:1: SyntaxWarning: invalid escape sequence '\.'
C:\Users\Robin\AppData\Local\Temp\ipykernel_11668\3505235198.py:1: SyntaxWarning: invalid escape sequence '\.'
  satzende_match = '(?<!\b(?:Dr|med|z\.B|etc)).\s+(?=[A-Z])'


# Sonderfälle

manche Redner wie z.B. amtierende Minister, Staatssekretäre etc. fallen aus dem Raster heraus und es steht keine Partei dahinter -> Wir wollen trotzdem die Parteien anmerken  
-> Abgleich mit XML-Stammdatenliste. Hier besonders angenehm: Schema: <p>"&lt;Vorname> &lt;Name>, &lt;Titel>:" </p> -> Das Ganze auch OHNE Doktortitel bei z.B. Dr. Robert Habeck -> Auslesen von Stammdaten xml